## 4.5 卷积层（Convolution Layer） - 多通道图像卷积计算

#### 1. 多通道图像是什么

##### 1.1 从灰度图到彩色图
我们前面举的很多例子，都是灰度图：
* 只有一个通道
* 形状通常写作：H × W × 1

但真实图像中，更常见的是彩色图像。

一张标准 RGB 彩色图像通常有 3 个通道：
* R（Red）红色通道
* G（Green）绿色通道
* B（Blue）蓝色通道

所以一张彩色图像的形状通常写作：

`H × W × 3`

例如一张 28 × 28 的彩色图片，可以写成：

`28 × 28 × 3`

##### 1.2 怎么理解“通道”
可以把通道理解为：

同一张图像，从不同颜色维度拆开的多层矩阵。

例如一张 RGB 图像，并不是一个单独的二维矩阵，

而是由 3 张大小相同的二维矩阵叠起来形成的：
* 一层记录红色强度
* 一层记录绿色强度
* 一层记录蓝色强度

所以，多通道图像本质上就是：

多个二维特征层叠在一起的输入数据 📚

##### 1.3 为什么要学习多通道卷积
因为在真实 CNN 中：
* 输入往往不是单通道
* 中间层输出的特征图也常常是多通道
* 所以卷积计算绝大多数时候其实都是多通道卷积

#### 2. 多通道卷积和单通道卷积的区别

##### 2.1 单通道卷积回顾
在单通道情况下：
* 输入是一张二维图
* 卷积核也是一个二维小矩阵
* 卷积核滑动后得到一张特征图

例如：
* 输入：28 × 28 × 1
* 卷积核：3 × 3
* 输出：一张二维特征图

##### 2.2 多通道时最大的变化
当输入变成多通道时，卷积核就不能再只是一个二维矩阵了。

它必须和输入的通道数对齐。

也就是说：

如果输入是 `H × W × 3`，那么一个卷积核的深度也必须是 `3`。

例如：
* 输入：28 × 28 × 3
* 卷积核大小：3 × 3

那么这个卷积核真实形状应该是：

`3 × 3 × 3`

这里最后那个 3，表示它要同时覆盖输入的 3 个通道。

##### 2.3 一个关键结论
>输入有几个通道，一个卷积核就必须有几个通道。

这是多通道卷积里最重要的规则之一。

你可以直接记成：

>卷积核的“深度”必须等于输入的通道数。

#### 3. 多通道卷积是如何计算的

##### 3.1 先同时取出所有通道的局部区域
假设输入是一张 RGB 图像，形状为：

`5 × 5 × 3`

现在使用一个卷积核，大小为：

`3 × 3 × 3`

当卷积核滑动到某个位置时，它不是只取某一个通道的 3 × 3 区域，

而是会同时取出：
* R 通道的一个 3 × 3
* G 通道的一个 3 × 3
* B 通道的一个 3 × 3

也就是说，此时取出的局部区域整体形状也是：

`3 × 3 × 3`

##### 3.2 每个通道分别做卷积计算
然后，卷积核也分成 3 层：
* 第 1 层和输入的 R 通道局部区域做逐元素乘法
* 第 2 层和输入的 G 通道局部区域做逐元素乘法
* 第 3 层和输入的 B 通道局部区域做逐元素乘法

也就是说：

先在每个通道内部，各自做一次“单通道卷积式”的乘加计算。

##### 3.3 再把三个通道的结果相加
当 3 个通道都分别算完后：
* 得到 R 通道一个和
* 得到 G 通道一个和
* 得到 B 通道一个和

最后再把它们加起来，通常再加上偏置 b：

`输出值 = R通道结果 + G通道结果 + B通道结果 + b`

这个最终结果，就是当前位置的一个输出值。

##### 3.4 结果解释
多通道卷积不是“分别输出 3 个值”，

而是：

每个通道先各算一部分，最后再加总，得到一个输出值。

这一点非常关键，

也就是说，特征图不是3层结构

#### 4. 一个完整的小例子

##### 4.1 输入是一个 3 × 3 × 3 局部区域

为了方便演示，假设当前位置取出的输入局部区域如下：

---
R 通道：
```
1 0 1
2 1 0
0 1 1
```
---
G 通道：
```
0 1 1
1 0 2
1 1 0
```
---
B 通道：
```
1 1 0
0 2 1
1 0 1
```

##### 4.2 卷积核也是 3 × 3 × 3

假设这个卷积核的三个通道分别是：

---

对应 R 的核：
```
1 0 1
0 1 0
1 0 1
```
---
对应 G 的核：
```
0 1 0
1 0 1
0 1 0
```
---
对应 B 的核：
```
1 0 0
0 1 0
0 0 1
```

##### 4.3 卷积计算
算 R 通道

对应位置相乘后求和：
```
1×1 + 0×0 + 1×1
+ 2×0 + 1×1 + 0×0
+ 0×1 + 1×0 + 1×1
= 1 + 0 + 1 + 0 + 1 + 0 + 0 + 0 + 1
= 4
```
---
算 G 通道
```
0×0 + 1×1 + 1×0
+ 1×1 + 0×0 + 2×1
+ 1×0 + 1×1 + 0×0
= 0 + 1 + 0 + 1 + 0 + 2 + 0 + 1 + 0
= 5
```
---
算 B 通道
```
1×1 + 1×0 + 0×0
+ 0×0 + 2×1 + 1×0
+ 1×0 + 0×0 + 1×1
= 1 + 0 + 0 + 0 + 2 + 0 + 0 + 0 + 1
= 4
```

##### 4.4 三个通道结果相加
最后把三个通道结果加起来：

`4 + 5 + 4 = 13`

如果偏置 b = 1，那么最终输出就是：

`13 + 1 = 14`

也就是说，这个卷积核在当前位置的输出值就是：

`14`

#### 5. 为什么 3 个通道最后只得到 1 个值

##### 5.1 因为这是“一个卷积核”的输出
* 输入有 3 个通道
* 卷积核也有 3 个通道
* 为什么最后不是输出 3 个值，而只输出 1 个值？

原因是：

一个卷积核整体只对应一种特征检测器。

虽然它内部有 3 个通道，但这 3 层不是独立输出的，

而是共同工作，一起检测某种模式。

所以最后：

>一个卷积核，不管输入有几个通道，最终都只会在一个位置输出一个值。

##### 5.2 这 1 个值表示什么
这个值表示：

当前这个卷积核，在当前位置上，对输入多通道局部区域的整体响应强度。

也可以理解为：

这个卷积核把 R、G、B 三个通道的信息融合起来后，

判断当前位置到底有多像它想找的特征。

#### 6. 多通道图像的卷积计算特征图结果
由于：

>一个卷积核，不管输入有几个通道，最终都只会在一个位置输出一个值。

所以：

>不管一个图像的通道有几个，经过一个卷积核的卷积计算的最终的特征图的shape都是：

`n * n`

比如：
* 输入图像 28 * 28 * 3
* kernel 3 * 3 * 3
* padding 1
* stride 1

最终特征图为：

`28 * 28`